In [4]:
import os
print(f"Database exists: {os.path.exists('resumewhip.db')}")
print(f"Database path: {os.path.abspath('resumewhip.db')}")
import sqlite3

Database exists: True
Database path: /Users/nicholasjoseph/PromptEngineering/api_work/resume_project/jobapp_prompter/resumewhip.db


In [3]:
def get_db_connection():
    """ Function to let FastAPI handle multiple requests to prevent db lockup"""
    conn = sqlite3.connect("resumewhip.db", check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

In [4]:
get_db_connection()

In [7]:
# def get_db_connection():
#     import os
#     db_path = os.path.abspath("resumewhip.db")
#     print(f"📂 Using database at: {db_path}")
#     return sqlite3.connect(db_path)

DATABASE_PATH = os.getenv("DATABASE_PATH", "resumewhip.db")

def get_db_connection():
    """ Function to let FastAPI handle multiple requests to prevent db lockup"""
    conn = sqlite3.connect(DATABASE_PATH, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

In [8]:
get_db_connection()

In [9]:
def init_database():
    """Initialize SQLite database with required tables"""
    # temp print verification
    print(f"🔍 Connecting to database at: {DATABASE_PATH}")
    conn = sqlite3.connect(DATABASE_PATH)
    cursor = conn.cursor()
    # email TEXT UNIQUE to tie users to email
    cursor.execute('''
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                user_id TEXT UNIQUE NOT NULL,
                email TEXT UNIQUE,
                subscription_status TEXT DEFAULT 'free',
                credits_remaining INTEGER DEFAULT 3,
                stripe_customer_id TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                last_payment_date TIMESTAMP
            )
        ''')
    
    # IP usage table (for rate limiting on free users)
    cursor.execute('''
            CREATE TABLE IF NOT EXISTS ip_usage (
                   ip TEXT NOT NULL,
                   date TEXT NOT NULL,
                   count INTEGER DEFAULT 0,
                   PRIMARY KEY (ip, date)
            )
        ''')
        
    conn.commit()
    conn.close()


In [10]:
init_database()

🔍 Connecting to database at: resumewhip.db


In [ ]:
def test_db_write():
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("INSERT INTO users (user_id, email) VALUES (?, ?)", 
                      ("test-123", "test@test.com"))
    conn.commit()
    conn.close()
    return "Test write complete"